# Sequential dilution, step 2 of 3 — recover from every subgroup pair

Pairs every subgroup at one concentration with every subgroup at the adjacent
concentration, giving `M x M` combinations per concentration pair, and runs the
recovery network independently on each. A single 10-fold dilution is the only
mixing diversity used.

**Input** — the per-virus subgroup folders from step 1, plus the DNA-probe
reference and the measured RNA reference.

**Output** — `extracted_spectra_final_epoch.csv` with one column per
combination, the g and h coefficient tables, and the metric CSVs.

**Feeds** — Fig. 3.

**Next** — `03_merge_extracted.ipynb`.

**Scale factor.** Spectra are multiplied by `SCALE_FACTOR = 400` before the network sees them and the factor is divided out again before anything is written, so every saved spectrum is on the mean-normalized scale. This is a numerical-conditioning choice only: the measured spectrum, the DNA reference and the ground-truth reference are all scaled by the same constant, so the recovered coefficients and every shape-based metric are unchanged.


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
import torch.optim as optim
import os
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from pathlib import Path
from itertools import product
import gc

In [ ]:
"""
HIERARCHICAL BATCH PROCESSING: Virus variants × Concentration pairs × Subgroup combinations
"""

# Root directory containing all variant subfolders
DATA_ROOT = "/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/03042026-train-K10_M10-subgroup_creation"

# Shared DNA reference
DNA_REF_PATH = "/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/ProbeDNA-trueRNA.csv"

# RNA reference paths for each variant
RNA_REF_PATHS = {
    'B1': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/B1-trueRNA.csv',
    'B1351': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/B1351-trueRNA.csv',
    'B16172': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/B16172-trueRNA.csv',
    'BA5': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/BA5-trueRNA.csv',
    'EG51': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/EG51-trueRNA.csv',
    'JN1': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/JN1-trueRNA.csv',
    'SARSCoV2': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/SARSCoV2-trueRNA.csv',
    'XBB15': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/XBB15-trueRNA.csv',
}

# Viruses to process (can select subset)
VIRUSES_TO_PROCESS = ['B1', 'B1351', 'B16172', 'BA5', 'EG51', 'JN1', 'SARSCoV2', 'XBB15']

# Concentration groups
CONCENTRATION_GROUPS = {
    'other_than_B1351': [
        98.0, 195.0, 391.0, 781.0, 1562.0, 3125.0, 6250.0, 12500.0, 25000.0, 50000.0, 100000.0
    ],
    'B1351': [
        191.0, 391.0, 781.0, 1562.0, 3125.0, 6250.0, 12500.0, 25000.0, 50000.0, 100000.0
    ],
}

# Number of subgroups per concentration
NUM_SUBGROUPS = 10

# Output configuration
OUTPUT_BASE = "/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/03042026-train-Yingchuan-udpated_SDE_M10_extraction"
SAVE_ALL_EPOCH_DATA = False  # Set to True to save full epoch-by-epoch data (large!)

# Training configuration
SCALE_FACTOR = 400.0
NUM_EPOCHS = 1000

os.makedirs(OUTPUT_BASE, exist_ok=True)

In [ ]:
def get_concentration_list(variant):
    """Get concentration list for a specific variant"""
    if variant == 'B1351':
        return CONCENTRATION_GROUPS['B1351']
    else:
        return CONCENTRATION_GROUPS['other_than_B1351']

def get_csv_path(variant, concentration, subgroup):
    """Construct path to subgroup average CSV"""
    filename = f"{variant}_{concentration}_subgroup{subgroup}_avg.csv"
    return os.path.join(DATA_ROOT, variant, filename)

def count_available_subgroups(variant, concentration):
    """
    Count how many subgroups actually exist for a given variant and concentration
    Returns the number of subgroups found (could be less than NUM_SUBGROUPS)
    """
    available_subgroups = []
    for sg in range(1, NUM_SUBGROUPS + 1):
        csv_path = get_csv_path(variant, concentration, sg)
        if os.path.exists(csv_path):
            available_subgroups.append(sg)
    return available_subgroups

def get_concentration_pairs(variant):
    """Get all consecutive concentration pairs for a variant"""
    conc_list = get_concentration_list(variant)
    pairs = [(conc_list[i], conc_list[i+1]) for i in range(len(conc_list)-1)]
    del conc_list
    return pairs

def get_combination_name(conc1, sg1, conc2, sg2):
    """Generate combination name: e.g., '98.0sg1_195.0sg2'"""
    return f"{conc1}sg{sg1}_{conc2}sg{sg2}"

def find_common_wavenumber_range(wavenumber_arrays):
    """
    Find the common wavenumber range across multiple arrays
    Returns (start_idx, end_idx, common_wavenumbers)
    
    Args:
        wavenumber_arrays: list of numpy arrays of wavenumbers
    """
    # Find minimum length
    min_length = min(len(arr) for arr in wavenumber_arrays)
    
    # Check if all arrays have the same values up to min_length
    # Truncate all to min_length
    truncated = [arr[:min_length] for arr in wavenumber_arrays]
    
    # Verify they're all the same (within floating point tolerance)
    reference = truncated[0]
    for i, arr in enumerate(truncated[1:], 1):
        if not np.allclose(reference, arr, rtol=1e-5, atol=1e-5):
            print(f"[Warning] Wavenumber mismatch detected in array {i}")
            # Find where they diverge
            diff_idx = np.where(~np.isclose(reference, arr, rtol=1e-5, atol=1e-5))[0]
            if len(diff_idx) > 0:
                print(f"  First difference at index {diff_idx[0]}: {reference[diff_idx[0]]} vs {arr[diff_idx[0]]}")
                # Use the shorter range
                min_length = min(min_length, diff_idx[0])
    
    # Return the common wavenumbers (truncated to safe range)
    common_wavenumbers = reference[:min_length]
    
    return 0, min_length, common_wavenumbers

def initialize_summary_csvs(output_dir, wavenumbers):
    """Initialize empty CSV files with headers"""
    # Extracted spectra CSV
    df_extracted = pd.DataFrame({'Wavenumbers': wavenumbers})
    df_extracted.to_csv(os.path.join(output_dir, "extracted_spectra_final_epoch.csv"), index=False)
    del df_extracted
    
    # g coefficients CSV
    df_g = pd.DataFrame(columns=['combination', 'g_c1', 'g_c2'])
    df_g.to_csv(os.path.join(output_dir, "g_coefficients_final_epoch.csv"), index=False)
    del df_g
    
    # h coefficients CSV
    df_h = pd.DataFrame(columns=['combination', 'h_c1', 'h_c2'])
    df_h.to_csv(os.path.join(output_dir, "h_coefficients_final_epoch.csv"), index=False)
    del df_h
    
    # Metrics: extracted vs true
    df_metrics_ext = pd.DataFrame(columns=['combination', 'R2', 'Cosine', 'Pearson'])
    df_metrics_ext.to_csv(os.path.join(output_dir, "metrics_extracted_vs_true.csv"), index=False)
    del df_metrics_ext
    
    # Metrics: recon vs exp (separate files for each metric)
    df_r2 = pd.DataFrame(columns=['combination', 'R2_c1', 'R2_c2'])
    df_r2.to_csv(os.path.join(output_dir, "metrics_recon_vs_exp_r2.csv"), index=False)
    del df_r2
    
    df_cos = pd.DataFrame(columns=['combination', 'Cosine_c1', 'Cosine_c2'])
    df_cos.to_csv(os.path.join(output_dir, "metrics_recon_vs_exp_cosine.csv"), index=False)
    del df_cos
    
    df_pear = pd.DataFrame(columns=['combination', 'Pearson_c1', 'Pearson_c2'])
    df_pear.to_csv(os.path.join(output_dir, "metrics_recon_vs_exp_pearson.csv"), index=False)
    del df_pear
    
    gc.collect()

def append_to_summary_csvs(output_dir, combination_name, results):
    """Append results from one extraction to summary CSVs"""
    
    # 1. Append extracted spectrum as new column
    csv_path = os.path.join(output_dir, "extracted_spectra_final_epoch.csv")
    df = pd.read_csv(csv_path)
    df[combination_name] = results['f_final'] / SCALE_FACTOR  # undo the x400 conditioning factor: saved spectra are mean-normalized
    df.to_csv(csv_path, index=False)
    del df
    
    # 2. Append g coefficients
    csv_path = os.path.join(output_dir, "g_coefficients_final_epoch.csv")
    df = pd.read_csv(csv_path)
    new_row = pd.DataFrame([{
        'combination': combination_name,
        'g_c1': results['g_final'][0],
        'g_c2': results['g_final'][1]
    }])
    df = pd.concat([df, new_row], ignore_index=True)
    df.to_csv(csv_path, index=False)
    del df, new_row
    
    # 3. Append h coefficients
    csv_path = os.path.join(output_dir, "h_coefficients_final_epoch.csv")
    df = pd.read_csv(csv_path)
    new_row = pd.DataFrame([{
        'combination': combination_name,
        'h_c1': results['h_final'][0],
        'h_c2': results['h_final'][1]
    }])
    df = pd.concat([df, new_row], ignore_index=True)
    df.to_csv(csv_path, index=False)
    del df, new_row
    
    # 4. Append extracted vs true metrics
    csv_path = os.path.join(output_dir, "metrics_extracted_vs_true.csv")
    df = pd.read_csv(csv_path)
    new_row = pd.DataFrame([{
        'combination': combination_name,
        'R2': results['metrics_ext']['r2'],
        'Cosine': results['metrics_ext']['cosine'],
        'Pearson': results['metrics_ext']['pearson']
    }])
    df = pd.concat([df, new_row], ignore_index=True)
    df.to_csv(csv_path, index=False)
    del df, new_row
    
    # 5. Append recon vs exp metrics (R2)
    csv_path = os.path.join(output_dir, "metrics_recon_vs_exp_r2.csv")
    df = pd.read_csv(csv_path)
    new_row = pd.DataFrame([{
        'combination': combination_name,
        'R2_c1': results['metrics_recon']['r2'][0],
        'R2_c2': results['metrics_recon']['r2'][1]
    }])
    df = pd.concat([df, new_row], ignore_index=True)
    df.to_csv(csv_path, index=False)
    del df, new_row
    
    # 6. Append recon vs exp metrics (Cosine)
    csv_path = os.path.join(output_dir, "metrics_recon_vs_exp_cosine.csv")
    df = pd.read_csv(csv_path)
    new_row = pd.DataFrame([{
        'combination': combination_name,
        'Cosine_c1': results['metrics_recon']['cosine'][0],
        'Cosine_c2': results['metrics_recon']['cosine'][1]
    }])
    df = pd.concat([df, new_row], ignore_index=True)
    df.to_csv(csv_path, index=False)
    del df, new_row
    
    # 7. Append recon vs exp metrics (Pearson)
    csv_path = os.path.join(output_dir, "metrics_recon_vs_exp_pearson.csv")
    df = pd.read_csv(csv_path)
    new_row = pd.DataFrame([{
        'combination': combination_name,
        'Pearson_c1': results['metrics_recon']['pearson'][0],
        'Pearson_c2': results['metrics_recon']['pearson'][1]
    }])
    df = pd.concat([df, new_row], ignore_index=True)
    df.to_csv(csv_path, index=False)
    del df, new_row
    
    gc.collect()

In [ ]:
def load_single_spectrum(csv_path):
    """Load a single spectrum CSV (wavenumbers + intensities)"""
    df = pd.read_csv(csv_path)
    wavenumbers = df.iloc[:, 0].values.astype(np.float32)
    intensities = df.iloc[:, 1].values.astype(np.float32)
    del df
    gc.collect()
    return wavenumbers, intensities

def load_two_spectra_with_alignment(csv_path1, csv_path2, dna_path, rna_path):
    """
    Load two spectra for extraction WITH LENGTH ALIGNMENT
    All spectra are truncated to the common wavenumber range
    Returns normalized data ready for training
    """
    # Load all spectra first
    dna_x, dna_y = load_single_spectrum(dna_path)
    rna_x, rna_y = load_single_spectrum(rna_path)
    x1, y1 = load_single_spectrum(csv_path1)
    x2, y2 = load_single_spectrum(csv_path2)
    
    # Find common wavenumber range
    start_idx, end_idx, common_wavenumbers = find_common_wavenumber_range([
        dna_x, rna_x, x1, x2
    ])
    
    # Truncate all spectra to common range
    dna_y = dna_y[start_idx:end_idx]
    rna_y = rna_y[start_idx:end_idx]
    x1 = x1[start_idx:end_idx]
    y1 = y1[start_idx:end_idx]
    x2 = x2[start_idx:end_idx]
    y2 = y2[start_idx:end_idx]
    
    # Delete the original x arrays we don't need
    del dna_x, rna_x
    
    # Verify all have same length now
    lengths = [len(dna_y), len(rna_y), len(y1), len(y2), len(x1), len(x2)]
    if len(set(lengths)) != 1:
        raise ValueError(f"Length mismatch after truncation: {lengths}")
    
    # Normalize DNA
    dna_y_mean = dna_y.mean()
    dna_y_normalized = dna_y / dna_y_mean
    del dna_y, dna_y_mean
    
    # Normalize RNA
    rna_y_mean = rna_y.mean()
    rna_y_normalized = rna_y / rna_y_mean
    del rna_y, rna_y_mean
    
    # Normalize spectra
    y1_mean = y1.mean()
    y2_mean = y2.mean()
    y1_normalized = y1 / y1_mean
    y2_normalized = y2 / y2_mean
    del y1, y2, y1_mean, y2_mean
    
    # CRITICAL: Use the TRUNCATED x1 and x2 for concatenation
    # Combine into dataset format
    x_combined = np.concatenate([x1, x2])
    y_combined = np.concatenate([y1_normalized, y2_normalized])
    s_combined = np.concatenate([
        np.zeros(len(x1)),
        np.ones(len(x2))
    ])
    del y1_normalized, y2_normalized, x1, x2
    
    # Create dataframe
    df = pd.DataFrame({
        'x': x_combined,
        'y': y_combined,
        's': s_combined
    })
    del x_combined, y_combined, s_combined
    
    gc.collect()
    return df, dna_y_normalized, rna_y_normalized, common_wavenumbers

In [ ]:
class FourierFeatureMapping(nn.Module):
    def __init__(self, num_frequencies=6, include_input=True):
        super().__init__()
        self.num_frequencies = num_frequencies
        self.include_input = include_input
        self.freq_bands = 2.0 ** torch.arange(0, num_frequencies).float() * np.pi
    
    def forward(self, x):
        out = [x] if self.include_input else []
        for freq in self.freq_bands.to(x.device):
            out.append(torch.sin(freq * x))
            out.append(torch.cos(freq * x))
        return torch.cat(out, dim=-1)

class ResBlock(nn.Module):
    def __init__(self, dim):
        super(ResBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )
        self.activation = nn.ReLU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class SERSDecomposition(nn.Module):
    def __init__(self, input_x_dim=13, input_s_dim=1, hidden_dim=256, z_dim=128): 
        super(SERSDecomposition, self).__init__()
        self.f_input = nn.Sequential(
            nn.Linear(input_x_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )   
        self.f_blocks = nn.Sequential(
            ResBlock(hidden_dim),
            ResBlock(hidden_dim)
        )
        self.f_output = nn.Sequential(
            nn.Linear(hidden_dim, z_dim),
            nn.ReLU(),
            nn.Linear(z_dim, 1)
        )
        self.c_input = nn.Sequential(
            nn.Linear(input_s_dim, hidden_dim),
            nn.ReLU()
        )
        self.c_blocks = nn.Sequential(
            ResBlock(hidden_dim),
            ResBlock(hidden_dim),
            ResBlock(hidden_dim)
        )
        self.c_output = nn.Sequential(
            nn.Linear(hidden_dim, 2),
            nn.Softplus() 
        )
    
    def forward(self, x_embed, s):
        fx = self.f_input(x_embed)
        fx = self.f_blocks(fx)
        f_x = self.f_output(fx)
        cs = self.c_input(s)
        cs = self.c_blocks(cs)
        a_c = self.c_output(cs)
        a_s = a_c[:, 0:1]
        b_s = a_c[:, 1:2]  # ← FIXED: was [:, 2:3] which is out of bounds!
        return a_s, b_s, f_x

def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [ ]:
def flatness_penalty(f_x, threshold, region_length=5):
    f_x = f_x.view(-1)
    diffs = f_x[1:] - f_x[:-1]
    sq_diffs = diffs ** 2
    sq_diffs = sq_diffs.unsqueeze(0).unsqueeze(0)
    kernel = torch.ones(1, 1, region_length, device=f_x.device) / region_length
    avg_sq = F.conv1d(sq_diffs, kernel, padding=region_length // 2).squeeze()
    penalty = torch.clamp(threshold - avg_sq, min=0.0)
    return torch.mean(penalty)

def custom_loss(y_true, a_s, b_s, f_x, bg, scale_mode=SCALE_FACTOR, 
                lambda_penalty=1.0, lambda_flat=0.05, threshold=1e-3, apply_flatness=False):
    threshold = threshold * (scale_mode ** 2)
    recon = a_s * f_x + b_s * bg
    mse_loss = torch.mean((y_true - recon) ** 2)
    neg_penalty = torch.mean(torch.clamp(-f_x, min=0.0)) * scale_mode 
    flat_pen = 0.0
    if apply_flatness:
        flat_pen = flatness_penalty(f_x, region_length=5, threshold=threshold)
    total_loss = (mse_loss + lambda_penalty * neg_penalty + lambda_flat * flat_pen)
    return total_loss

In [ ]:
def to_1d(x):
    return x.detach().cpu().numpy().reshape(-1)

def train_two_spectra(df, bg_spectrum, true_spectrum, wavenumbers, device='cuda', num_epochs=1000, verbose=False):
    """
    Train on exactly two spectra
    Returns final epoch results only
    
    Args:
        df: DataFrame with x, y, s columns
        bg_spectrum: Background spectrum (normalized)
        true_spectrum: True RNA spectrum (normalized)
        wavenumbers: Original wavenumber values for output
        device: 'cuda' or 'cpu'
        num_epochs: Number of training epochs
        verbose: Print progress
    """
    # Prepare data
    x = df[['x']].values.astype(np.float32)
    y = df[['y']].values.astype(np.float32)
    s_raw = df['s'].values
    del df  # Delete DataFrame after extracting arrays
    
    s_train = torch.tensor(s_raw.astype(np.float32).reshape(-1, 1)).to(device)
    del s_raw
    
    bg = np.tile(bg_spectrum, (len(y)//len(bg_spectrum), 1)).reshape(-1, 1)
    x_train = torch.tensor(x, dtype=torch.float32).to(device)
    y_train = torch.tensor(y, dtype=torch.float32).to(device)
    bg_train = torch.tensor(bg, dtype=torch.float32).to(device)
    del x, y, bg
    
    # Scaling
    y_train = y_train * SCALE_FACTOR
    bg_train = bg_train * SCALE_FACTOR
    
    # Normalize x
    x_min = x_train.min()
    x_max = x_train.max()
    x_train = (x_train - x_min) / (x_max - x_min)
    del x_min, x_max
    
    # Feature mapping
    fourier_mapping = FourierFeatureMapping(num_frequencies=6).to(device)
    x_embed_train = fourier_mapping(x_train)
    
    # Model
    input_x_dim = x_embed_train.shape[1]
    model = SERSDecomposition(input_x_dim=input_x_dim).to(device)
    model.apply(weights_init)
    
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)
    
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        apply_flatness = epoch >= 500
        
        a_s, b_s, f_x = model(x_embed_train, s=s_train)
        loss = custom_loss(y_train, a_s, b_s, f_x, bg_train, 
                          scale_mode=SCALE_FACTOR, apply_flatness=apply_flatness)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()
        scheduler.step()
        
        if verbose and (epoch + 1) % 200 == 0:
            print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.6f}")
    
    # Extract final epoch results
    model.eval()
    with torch.no_grad():
        a_s, b_s, f_x = model(x_embed_train, s=s_train)
    
    # Delete model to free GPU memory
    del model, optimizer, scheduler, fourier_mapping
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Convert to numpy
    s_train_np = to_1d(s_train)
    bg_train_np = to_1d(bg_train)
    f_x_np = to_1d(f_x)
    a_s_np = to_1d(a_s)
    b_s_np = to_1d(b_s)
    y_train_np = to_1d(y_train)
    
    # Delete tensors
    del s_train, bg_train, f_x, a_s, b_s, y_train, x_train, x_embed_train
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    unique_s = np.unique(s_train_np)
    mask0 = (s_train_np == unique_s[0]).squeeze()
    
    f0 = f_x_np[mask0]
    bg0 = bg_train_np[mask0]
    x0 = wavenumbers  # Use original wavenumbers
    
    # Area normalization
    f_area = np.trapz(f0, x=x0, axis=0)
    bg_area = np.trapz(bg0, x=x0, axis=0)
    
    f_final = f0 * (bg_area / f_area)
    bg_final = bg0
    true_final = true_spectrum * SCALE_FACTOR
    
    del f0, bg0, bg_train_np, f_x_np
    
    # Metrics: extracted vs true
    r2_ext = r2_score(true_final, f_final)
    cos_ext = 1 - cosine(true_final, f_final)
    pear_ext, _ = pearsonr(true_final, f_final)
    
    # Get coefficients for both spectra
    g_final = []
    h_final = []
    r2_recon = []
    cos_recon = []
    pear_recon = []
    y_exp_list = []
    recon_list = []
    
    for s_val in unique_s:
        mask = (s_train_np == s_val).squeeze()
        y_sub = y_train_np[mask]
        
        a_pred_sub = a_s_np[mask][0]
        b_pred_sub = b_s_np[mask][0]
        
        a_final = a_pred_sub * (f_area / bg_area)
        b_final = b_pred_sub
        
        g_final.append(a_final)
        h_final.append(b_final)
        
        recon_np = a_final * f_final + b_final * bg_final
        
        r2_recon.append(r2_score(y_sub, recon_np))
        cos_recon.append(1 - cosine(y_sub, recon_np))
        pear, _ = pearsonr(y_sub, recon_np)
        pear_recon.append(pear)
        
        y_exp_list.append(y_sub)
        recon_list.append(recon_np)
    
    del s_train_np, a_s_np, b_s_np, y_train_np, mask, unique_s
    gc.collect()
    
    return {
        'x0': x0,
        'f_final': f_final,
        'g_final': np.array(g_final),
        'h_final': np.array(h_final),
        'bg_final': bg_final,
        'true_final': true_final,
        'y_exp_list': y_exp_list,
        'recon_list': recon_list,
        'metrics_ext': {
            'r2': r2_ext,
            'cosine': cos_ext,
            'pearson': pear_ext
        },
        'metrics_recon': {
            'r2': r2_recon,
            'cosine': cos_recon,
            'pearson': pear_recon
        }
    }

In [ ]:
def save_reconstruction_figures(results, output_dir, variant, conc1, sg1, conc2, sg2, combination_name):
    """
    Save 2 reconstruction figures (one per spectrum)
    CRITICAL: Proper order is savefig() -> close() -> clf() -> cla()
    """
    fig_dir = os.path.join(output_dir, "figures_reconstruction")
    os.makedirs(fig_dir, exist_ok=True)
    
    x0 = results['x0']
    
    # Figure for concentration 1
    plt.figure(figsize=(7, 5))
    plt.plot(x0, results['y_exp_list'][0], 'r-', lw=2, label='Experimental')
    plt.plot(x0, results['recon_list'][0], 'g--', lw=2, label='Reconstructed')
    plt.xlabel('Wavenumber (cm⁻¹)')
    plt.ylabel('Intensity (scaled)')
    plt.title(
        f"{variant} | Conc: {conc1} | Subgroup: {sg1}\n"
        f"R²={results['metrics_recon']['r2'][0]:.4f}, "
        f"Cos={results['metrics_recon']['cosine'][0]:.4f}, "
        f"Pearson={results['metrics_recon']['pearson'][0]:.4f}"
    )
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    fname = f"{combination_name}_recon_vs_exp_c1.png"
    plt.savefig(os.path.join(fig_dir, fname), dpi=300)
    # Proper cleanup order: close() -> clf() -> cla()
    plt.close()
    plt.clf()
    plt.cla()
    
    # Figure for concentration 2
    plt.figure(figsize=(7, 5))
    plt.plot(x0, results['y_exp_list'][1], 'r-', lw=2, label='Experimental')
    plt.plot(x0, results['recon_list'][1], 'g--', lw=2, label='Reconstructed')
    plt.xlabel('Wavenumber (cm⁻¹)')
    plt.ylabel('Intensity (scaled)')
    plt.title(
        f"{variant} | Conc: {conc2} | Subgroup: {sg2}\n"
        f"R²={results['metrics_recon']['r2'][1]:.4f}, "
        f"Cos={results['metrics_recon']['cosine'][1]:.4f}, "
        f"Pearson={results['metrics_recon']['pearson'][1]:.4f}"
    )
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    fname = f"{combination_name}_recon_vs_exp_c2.png"
    plt.savefig(os.path.join(fig_dir, fname), dpi=300)
    plt.close()
    plt.clf()
    plt.cla()
    
    gc.collect()

def save_extracted_figure(results, dna_spectrum, output_dir, variant, conc1, sg1, conc2, sg2, combination_name):
    """
    Save extracted vs references figure
    CRITICAL: Proper order is savefig() -> close() -> clf() -> cla()
    """
    fig_dir = os.path.join(output_dir, "figures_extracted")
    os.makedirs(fig_dir, exist_ok=True)
    
    x0 = results['x0']
    dna_scaled = dna_spectrum * SCALE_FACTOR
    
    plt.figure(figsize=(8, 5))
    plt.plot(x0, results['f_final'], 'g-', lw=2, label='Extracted')
    plt.plot(x0, results['true_final'], 'm--', lw=2, label='RNA Reference')
    plt.plot(x0, dna_scaled, 'y--', lw=1.5, alpha=0.7, label='DNA Reference')
    plt.xlabel('Wavenumber (cm⁻¹)')
    plt.ylabel('Intensity (scaled)')
    plt.title(
        f"{variant} | Combination: {combination_name}\n"
        f"g_c1={results['g_final'][0]:.3f}, g_c2={results['g_final'][1]:.3f} | "
        f"h_c1={results['h_final'][0]:.3f}, h_c2={results['h_final'][1]:.3f}\n"
        f"R²={results['metrics_ext']['r2']:.4f}, "
        f"Cos={results['metrics_ext']['cosine']:.4f}, "
        f"Pearson={results['metrics_ext']['pearson']:.4f}"
    )
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    fname = f"{combination_name}_extracted_vs_ref.png"
    plt.savefig(os.path.join(fig_dir, fname), dpi=300)
    plt.close()
    plt.clf()
    plt.cla()
    
    del dna_scaled
    gc.collect()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

total_variants = len(VIRUSES_TO_PROCESS)
variant_counter = 0

for variant in VIRUSES_TO_PROCESS:
    variant_counter += 1
    print(f"\n{'#'*80}")
    print(f"# [{variant_counter}/{total_variants}] Processing Variant: {variant}")
    print(f"{'#'*80}")
    
    # Get RNA reference for this variant
    rna_ref_path = RNA_REF_PATHS[variant]
    
    # Get concentration pairs
    conc_pairs = get_concentration_pairs(variant)
    print(f"Concentration pairs: {len(conc_pairs)}")
    
    for pair_idx, (conc1, conc2) in enumerate(conc_pairs, 1):
        print(f"\n{'='*80}")
        print(f"[{pair_idx}/{len(conc_pairs)}] Processing pair: {conc1} vs {conc2}")
        print(f"{'='*80}")
        
        # Count available subgroups for both concentrations
        subgroups_c1 = count_available_subgroups(variant, conc1)
        subgroups_c2 = count_available_subgroups(variant, conc2)
        
        if len(subgroups_c1) == 0 or len(subgroups_c2) == 0:
            print(f"⚠ Skipping pair {conc1} vs {conc2}: No subgroups found")
            print(f"  Conc {conc1}: {len(subgroups_c1)} subgroups")
            print(f"  Conc {conc2}: {len(subgroups_c2)} subgroups")
            continue
        
        print(f"Available subgroups:")
        print(f"  Conc {conc1}: {len(subgroups_c1)} subgroups {subgroups_c1}")
        print(f"  Conc {conc2}: {len(subgroups_c2)} subgroups {subgroups_c2}")
        
        # Create output directory
        pair_name = f"conc_{conc1}_vs_{conc2}"
        pair_output_dir = os.path.join(OUTPUT_BASE, variant, pair_name)
        os.makedirs(pair_output_dir, exist_ok=True)
        
        # Get a sample spectrum to determine common wavenumber range
        # We need to check DNA, RNA, and at least one sample from each concentration
        sample_csv1 = get_csv_path(variant, conc1, subgroups_c1[0])
        sample_csv2 = get_csv_path(variant, conc2, subgroups_c2[0])
        
        try:
            # Load samples to determine common range
            dna_x_sample, _ = load_single_spectrum(DNA_REF_PATH)
            rna_x_sample, _ = load_single_spectrum(rna_ref_path)
            x1_sample, _ = load_single_spectrum(sample_csv1)
            x2_sample, _ = load_single_spectrum(sample_csv2)
            
            # Find common range
            _, _, common_wavenumbers = find_common_wavenumber_range([
                dna_x_sample, rna_x_sample, x1_sample, x2_sample
            ])
            
            del dna_x_sample, rna_x_sample, x1_sample, x2_sample
            gc.collect()
            
            print(f"Common wavenumber range: {common_wavenumbers[0]:.1f} - {common_wavenumbers[-1]:.1f} cm⁻¹ ({len(common_wavenumbers)} points)")
            
        except Exception as e:
            print(f"✗ Error determining wavenumber range: {str(e)}")
            continue
        
        # Initialize summary CSVs with common wavenumbers
        initialize_summary_csvs(pair_output_dir, common_wavenumbers)
        
        # Process all available combinations
        total_combinations = len(subgroups_c1) * len(subgroups_c2)
        combination_counter = 0
        
        for sg1 in subgroups_c1:
            for sg2 in subgroups_c2:
                combination_counter += 1
                combination_name = get_combination_name(conc1, sg1, conc2, sg2)
                
                print(f"  [{combination_counter}/{total_combinations}] {combination_name}...", end=" ", flush=True)
                
                try:
                    # Get CSV paths
                    csv1 = get_csv_path(variant, conc1, sg1)
                    csv2 = get_csv_path(variant, conc2, sg2)
                    
                    # Verify files exist
                    if not os.path.exists(csv1):
                        print(f"✗ File not found: {csv1}")
                        continue
                    if not os.path.exists(csv2):
                        print(f"✗ File not found: {csv2}")
                        continue
                    
                    # Load data with alignment
                    df, dna_norm, rna_norm, aligned_wavenumbers = load_two_spectra_with_alignment(
                        csv1, csv2, DNA_REF_PATH, rna_ref_path
                    )
                    
                    # Verify aligned wavenumbers match common wavenumbers
                    if len(aligned_wavenumbers) != len(common_wavenumbers):
                        print(f"✗ Length mismatch: {len(aligned_wavenumbers)} vs {len(common_wavenumbers)}")
                        del df, dna_norm, rna_norm, aligned_wavenumbers
                        continue
                    
                    # Train
                    results = train_two_spectra(
                        df, dna_norm, rna_norm, aligned_wavenumbers,
                        device=device,
                        num_epochs=NUM_EPOCHS,
                        verbose=False
                    )
                    del df, dna_norm, rna_norm, aligned_wavenumbers
                    
                    # Save figures
                    save_reconstruction_figures(
                        results, pair_output_dir, variant,
                        conc1, sg1, conc2, sg2, combination_name
                    )
                    
                    # For extracted figure, we need dna_spectrum aligned to common range
                    dna_x_full, dna_y_full = load_single_spectrum(DNA_REF_PATH)
                    _, end_idx, _ = find_common_wavenumber_range([dna_x_full, common_wavenumbers])
                    dna_y_aligned = dna_y_full[:end_idx] / dna_y_full[:end_idx].mean()
                    del dna_x_full, dna_y_full
                    
                    save_extracted_figure(
                        results, dna_y_aligned, pair_output_dir, variant,
                        conc1, sg1, conc2, sg2, combination_name
                    )
                    del dna_y_aligned
                    
                    # Append to summary CSVs
                    append_to_summary_csvs(pair_output_dir, combination_name, results)
                    
                    # Delete results after saving
                    del results
                    gc.collect()
                    
                    print("✓")
                    
                except Exception as e:
                    print(f"✗ Error: {str(e)}")
                    import traceback
                    traceback.print_exc()
                    gc.collect()
                    continue
        
        # Clean up after this concentration pair
        del common_wavenumbers, subgroups_c1, subgroups_c2
        gc.collect()
        
        print(f"\n✓ Completed pair: {conc1} vs {conc2} ({combination_counter} combinations processed)")
    
    # Clean up after variant
    del conc_pairs
    gc.collect()

print("\n" + "="*80)
print("ALL PROCESSING COMPLETED!")
print("="*80)